<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — Preprocesamiento y Data Augmentation (v2.1)
**Proyecto:** Detección de Deslizamientos mediante IA | **EAFIT** 
**Estado:** Corregido (Deep Copy de Tensores).

In [ ]:
import os, sys, h5py, torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

DATA_ROOT = Path('/content/landslide4sense')
img_dir = DATA_ROOT / "TrainData" / "img"
mask_dir = DATA_ROOT / "TrainData" / "mask"
img_list = sorted(list(img_dir.glob("*.h5")))
mask_list = sorted(list(mask_dir.glob("*.h5")))

CHANNEL_NAMES = ['SAR VV', 'SAR VH', 'SAR VV/VH', 'B2-Blue', 'B3-Green', 'B4-Red', 
                 'B5-RE1', 'B6-RE2', 'B7-RE3', 'B8-NIR', 'B8A-RE4', 'B11-SWIR1', 'B12-SWIR2', 'DEM']

## 1. Clase Dataset (Fix Definitivo de Strides)
Para evitar el `ValueError`, realizamos el `.transpose()` y luego el `.copy()`. Esto garantiza que los datos sean contiguos en memoria antes de pasarlos a PyTorch.

In [ ]:
def zscore_norm(patch):
    mean = np.mean(patch, axis=(0, 1))
    std = np.std(patch, axis=(0, 1)) + 1e-6
    return (patch - mean) / std

class LandslideDataset(Dataset):
    def __init__(self, img_files, mask_files, augment=False):
        self.img_files = img_files
        self.mask_files = mask_files
        self.augment = augment

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        with h5py.File(self.img_files[idx], 'r') as hf: 
            img = hf[list(hf.keys())[0]][()].astype(np.float32)
        with h5py.File(self.mask_files[idx], 'r') as hf: 
            mask = hf[list(hf.keys())[0]][()].astype(np.float32)
        
        # 1. Normalización
        img = zscore_norm(img)
        
        # 2. Augmentación
        if self.augment:
            k = np.random.randint(0, 4)
            img = np.rot90(img, k)
            mask = np.rot90(mask, k)
            
            if np.random.random() > 0.5:
                img = np.flip(img, axis=1)
                mask = np.flip(mask, axis=1)

        # 3. Cambio de ejes [H, W, C] -> [C, H, W]
        img = img.transpose(2, 0, 1)
        
        # 4. CRÍTICO: .copy() DESPUÉS del transpose para asegurar memoria contigua
        img = np.ascontiguousarray(img)
        mask = np.ascontiguousarray(mask)

        # 5. Conversión a Tensor
        return torch.from_numpy(img).float(), torch.from_numpy(mask).long()

## 2. Validación de la Pipeline
Inspección visual de 4 transformaciones aleatorias del mismo parche.

In [ ]:
ds = LandslideDataset(img_list, mask_list, augment=True)
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
plt.style.use('seaborn-v0_8-whitegrid')

for i in range(4):
    # Muestra con deslizamiento
    idx_test = 1816 if len(ds) > 1816 else 0
    img, mask = ds[idx_test]
    
    # RGB Visual (Bands 5, 4, 3)
    rgb = img[[5, 4, 3]].numpy().transpose(1, 2, 0)
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-6)
    
    axes[0, i].imshow(rgb)
    axes[0, i].set_title(f"Augmentación {i+1}", fontweight='bold')
    
    axes[1, i].imshow(mask, cmap='inferno')
    axes[1, i].set_title("Mask (Label)", color='orange')
    
    for ax in axes[:, i]: ax.axis('off')

plt.tight_layout()
plt.show()